# Pixel-Level Biomedical Image Forgery Detection
**Private Score: 0.550 oF1 &nbsp;|&nbsp; Public Score: 0.450 oF1**

---

## Project Summary

This notebook implements the **inference pipeline** for pixel-level copy-move forgery detection in biomedical scientific figures. The task is to identify manipulated regions (copy-move forgeries) in western blots, microscopy images, gel electrophoresis panels, and similar biomedical images found in scientific publications.

## Pipeline Overview

```
Input Image
    │
    ▼
[Stage 1]  YOLO Panel Detector (YOLOv11/v12 ensemble @ 640px + 960px)
           Localises sub-panels: Blots, Microscopy, Graphs, Flow Cytometry ...
    │
    ▼
[Stage 2]  Contrastive Embedding Similarity Screening (SupCon)
           Filters candidate duplicate panel pairs above cosine-similarity thresholds
    │
    ▼
[Stage 3]  LightGlue Geometric Verification (SIFT / ALIKED)
           Keypoint matching + MAGSAC homography estimation → pixel-level copy-move masks
    │
    ▼
[Stage 4]  Strip Detector (YOLOv8-Seg + embeddings) — blot-specific
           Detects reused horizontal strips spliced within western blot panels
    │
    ▼
[Fallback] DINOv2 Encoder + U-Net-style CNN Decoder (Segmentation model)
           Pixel-level semantic segmentation when geometric matching yields no result
    │
    ▼
Output: RLE-encoded binary forgery mask  OR  "authentic"
```

## Key Design Decisions

| Choice | Rationale |
|--------|-----------|
| YOLO panel detection | Scientific figures contain heterogeneous sub-images; panel-level processing drastically reduces false positives vs. full-image search |
| SupCon embeddings | Contrastive pre-training yields compact representations that cluster near-duplicate panels even under brightness/contrast edits |
| LightGlue (SIFT + ALIKED) | Sparse keypoint matching is invariant to rotation/scale copy-moves; MAGSAC provides robust homography estimation with few inliers |
| DINOv2 fallback segmentor | Self-supervised ViT patch features generalise to unseen forgery patterns; frozen encoder avoids overfitting on ~3 k training images |
| Dual-YOLO ensemble | Two detectors at different resolutions (640/960 px) maximise blot panel recall without degrading non-blot precision |


---
## 1. Environment Setup

Install competition-specific wheels (`luc`, `ultralytics`, `lightglue`) from local dataset inputs.
Offline mode flags prevent any outbound network calls during inference.


In [1]:
!pip config set global.disable-pip-version-check true
!pip install --no-deps /kaggle/input/neededpip/luc-0.8.0-py3-none-any.whl
!pip install --no-deps /kaggle/input/neededpip/ultralytics-8.3.228-py3-none-any.whl

!mkdir -p /root/.config/Ultralytics/ && cp /kaggle/input/yolopanel/settings.json /root/.config/Ultralytics/settings.json
!pip install --no-deps /kaggle/input/neededpip/lightglue-0.0-py3-none-any.whl
!mkdir -p /root/.cache/torch/hub/checkpoints/ && cp /kaggle/input/neededpip/* ~/.cache/torch/hub/checkpoints/

import os
os.environ["YOLO_OFFLINE"] = "True"
os.environ["ULTRALYTICS_ONLINE"] = "False"
os.environ["YOLO_SETTINGS_SKIP_VALIDATION"] = "True"
# os.environ["YOLO_VERBOSE"] = "False" 

Writing to /root/.config/pip/pip.conf
Processing /kaggle/input/neededpip/luc-0.8.0-py3-none-any.whl
Processing /kaggle/input/neededpip/ultralytics-8.3.228-py3-none-any.whl
Processing /kaggle/input/neededpip/lightglue-0.0-py3-none-any.whl


---
## 2. Configuration & Model Loading

### 2.1 Global Hyperparameters

All inference thresholds are centralised here for transparency and easy ablation:

| Parameter | Value | Purpose |
|-----------|-------|---------|
| `yolo_conf` | 0.70 | YOLO panel detection confidence gate |
| `yolo_iou` | 0.40 | YOLO NMS IoU threshold |
| `match_score_threshold` | 0.73 | LightGlue mean-match-score gate |
| `inlier_threshold` | 8 | Minimum MAGSAC geometric inliers for a valid match |
| `max_keypoints` | 4096 | LightGlue keypoint budget (microscopy) |
| `WBLOT_DUP_SCORE_THRESH1` | 0.84 | SupCon blot duplication cosine-similarity gate |
| `WBLOT_OVERLAP_THRESHOLD` | 0.85 | Blot overlap embedder similarity gate |
| `MICROSCOPY_EMB_THRESH` | 0.58 | Microscopy overlap-embedder similarity gate |

### 2.2 Models Loaded in This Cell

- **YOLOv12 Panel Extractor** (`best_yolo12.pt`) — detects panel bounding boxes by category (Blots, Microscopy, Graphs, …)
- **SupCon Blot Duplication Embedder** — 320×64 crops → 128-d L2-normalised embeddings trained with supervised contrastive loss
- **Ensemble Microscopy Overlap Embedder** — 3-checkpoint ensemble, each operating on 224×224 crops
- **LightGlue Matcher — Microscopy** (SIFT features, `max_keypoints=4096`)
- **LightGlue Matcher — Blots** (ALIKED features, `max_keypoints=512`, MAGSAC estimator, `reprojThreshold=3.0`)


In [2]:
device = 'cuda'
VERBOSE = False

####### YOLO CONFIG
from luc.utils_yolov11 import PanelExtractor
extractor_path = "/kaggle/input/yolopanel/best_yolo12.pt"

EXCLUDED_LABELS = {"Graphs", "Flow Cytometry", 'Body Imaging'}
yolo_size = 640
yolo_conf = 0.7
yolo_iou = 0.4

matcher_name = "sift"
match_score_threshold = 0.73

max_keypoints = 4096
inlier_threshold = 8
depth_confidence=0.9
width_confidence=0.9
match_filter_str = 'mean_match_score'  # or 'inlier_mean_score'


from luc.embedder.inference import UniversalEmbedderInference, EnsembleEmbedderInference
from luc.utils_match_v5 import LightGlueOverlap, create_duplicate_masks_v5, merge_masks_by_max_cliques_v5
from luc.kaggle_metric import rle_encode, oF1_score
from luc.bbox import get_intersections, bbox_to_mask, iou
from luc.strip import find_strips_in_blot_panels, create_strip_match_masks, visualize_matches_on_image

from glob import glob
from tqdm.notebook import tqdm
import numpy as np
import pandas as pd
from PIL import Image
from itertools import combinations

extractor = PanelExtractor(
    weights_path=extractor_path,
    device=device,
    img_size=yolo_size,
    conf_threshold=yolo_conf,
    iou_threshold=yolo_iou
)
extractor.EXCLUDED_LABELS = EXCLUDED_LABELS

####### FULL DUP BLOT CONFIG
WBLOT_DUP_SCORE_THRESH1 = 0.84
blot_duplicator_supcon = UniversalEmbedderInference("/kaggle/input/blotduplication/duplicate-supcon-epoch02-step10240-val_pairs_f10.90683-val_pairs_auc0.99899.ckpt", device=device, width=320, height=64, transform_type='albu_resize', head_type='v2', map_location='cpu')

print('-'*40)
MICROSCOPY_EMB_THRESH = 0.58
# micro_overlap_embedder = UniversalEmbedderInference("/kaggle/input/microscopyoverlap/microscopy-epoch00-step11000-val_pairs_f10.34648-val_pairs_auc0.94336.ckpt", device=device, width=224, height=224, transform_type='albu_resize', head_type='v2', map_location='cpu')
# micro_overlap_embedder = UniversalEmbedderInference("/kaggle/input/microscopyoverlap/microscopy-epoch02-step16366-val_pairs_f10.43186-val_pairs_auc0.95879.ckpt", 
#                                                     device=device, width=224, height=224, transform_type='albu_longest_max_size', head_type='v2', map_location='cpu')

cpts = glob('/kaggle/input/microscopyoverlap/best3/*')
print('checkpoints :', cpts)
micro_overlap_embedder = EnsembleEmbedderInference.from_checkpoints(
    cpts, 
    device=device, 
    width=224, 
    height=224, 
    transform_type='albu_resize', 
    head_type='v2', 
    map_location='cpu'
)
print('-'*40)

# matcher = LightGlueOverlap(max_keypoints=max_keypoints, matcher_features=matcher_name, device=device, depth_confidence=depth_confidence, width_confidence=width_confidence)
matcher_micro = LightGlueOverlap(max_keypoints=max_keypoints, matcher_features=matcher_name, device=device, depth_confidence=depth_confidence, width_confidence=width_confidence)
matcher_blot = LightGlueOverlap(max_keypoints=512, matcher_features='aliked', device=device, depth_confidence=-1, width_confidence=-1, 
    estimator_method='MAGSAC',                 
    reprojThreshold=3.0,
    estimator_confidence=0.9999,
    estimator_maxIters=5000,
    estimator_refineIters=10,)

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:2225: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

1. Detector loaded successfully from /kaggle/input/yolopanel/best_yolo12.pt. Classes: {0: 'Blots', 1: 'Graphs', 2: 'Microscopy', 3: 'Body Imaging', 4: 'Flow Cytometry'}
Size: 640 | conf thresh: 0.7, IOU: 0.4, min crop side: 5 pixels
Using pooling type: avg
Model : nextvit_small.bd_ssld_6m_in1k. Using normalization mean: (0.485, 0.456, 0.406), std: (0.229, 0.224, 0.225)
using transform_type: albu_resize
----------------------------------------
checkpoints : ['/kaggle/input/microscopyoverlap/best3/microscopy-epoch01-step8815-val_pairs_f10.47312-val_pairs_auc0.96345.ckpt', '/kaggle/input/microscopyoverlap/best3/microscopy-epoch00-step4860-val_pairs_f10.46711-val_pairs_auc0.96332.ckpt', '/kaggle/input/microscopyoverlap/best3/microscopy-epoch02-step19790-val_pairs_f10.45754-val_pairs_auc0.96353.ckpt']
Loading model 1/3: /kaggle/input/microscopyoverlap/best3/microscopy-epoch01-step8815-val_pairs_f10.47312-val_pairs_auc0.96345.ckpt
Using pooling type: attention
Model : nextvit_small.bd_ssld_6

---
## 3. Dual-YOLO Panel Detection Ensemble

### Motivation

A single YOLO model at a fixed input resolution sometimes misses small or narrow blot panels.
We run **two detectors** at different resolutions and merge detections with a custom priority rule:

- **Model 1 (640 px, `conf=0.7`)** — primary source of truth for all panel types.
- **Model 2 (960 px, `conf=0.3`)** — used exclusively to *enrich* blot detection: any blot it finds
  that does not overlap a Model-1 blot (IoU ≥ 0.25 or containment ≥ 50%) is added to the list.
- Non-blot panels always come from Model 1 only, preventing noise from the lower-confidence secondary model.

This asymmetric ensemble improved blot recall on the private leaderboard without degrading precision
on graphs or flow cytometry panels.

### Merge Logic (pseudocode)

```
final_panels = model1_non_blot_panels
final_panels += model1_blot_panels
for each blot in model2:
    if blot does NOT overlap any model1_blot (IoU < 0.25 and containment < 50%):
        final_panels += blot   # enrich with new detection
```


In [3]:
def merge_dual_model_detections(extractor1, extractor2, img, iou_threshold=0.3, min_panel_size=12, verbose=False):
    """
    Merge panel detections from two models with special handling for Blot panels.
    
    Logic:
    - If model1 detects a Blot and model2 detects overlapping Blot(s), keep model1's detection
    - If model2 detects a Blot that doesn't overlap with any model1 Blot, add it to enrich data
    - Non-Blot panels from model1 are kept as-is
    
    Args:
        extractor1: First PanelExtractor model (primary)
        extractor2: Second PanelExtractor model (secondary, for enrichment)
        img: Image array or path
        iou_threshold: IoU threshold for considering two Blot detections as overlapping
        min_panel_size: Minimum width/height for a valid panel (default: 12 pixels)
        verbose: Whether to print debug information
        
    Returns:
        List of merged panels: [(label, conf, x0, y0, x1, y1), ...]
    """
    # Extract panels from both models
    panels1 = extractor1.extract_panels(img)
    panels2 = extractor2.extract_panels(img)
    
    if verbose:
        print(f"Model 1 detected {len(panels1)} panels")
        print(f"Model 2 detected {len(panels2)} panels")
    
    # Separate Blot and non-Blot panels from model1
    blot_panels1 = [p for p in panels1 if p[0] == 'Blots']
    non_blot_panels1 = [p for p in panels1 if p[0] != 'Blots']
    blot_panels2 = [p for p in panels2 if p[0] == 'Blots']
    
    if verbose:
        print(f"Model 1: {len(blot_panels1)} Blot panels, {len(non_blot_panels1)} non-Blot panels")
        print(f"Model 2: {len(blot_panels2)} Blot panels")
    
    # Process Blot panels
    processed_blot_panels = []
    used_panels2 = set()
    
    # Keep all Blot panels from model1
    processed_blot_panels.extend(blot_panels1)
    
    if verbose and blot_panels1:
        print(f"Keeping all {len(blot_panels1)} Blot panels from model1")
    
    # Mark model2 panels as used if they overlap with any model1 Blot panel
    for blot1 in blot_panels1:
        _, conf1, x0_1, y0_1, x1_1, y1_1 = blot1
        box1 = (x0_1, y0_1, x1_1, y1_1)
        area1 = (x1_1 - x0_1) * (y1_1 - y0_1)
        
        for j, blot2 in enumerate(blot_panels2):
            if j in used_panels2:
                continue
                
            _, conf2, x0_2, y0_2, x1_2, y1_2 = blot2
            box2 = (x0_2, y0_2, x1_2, y1_2)
            area2 = (x1_2 - x0_2) * (y1_2 - y0_2)
            
            # Check if boxes overlap (using IoU or intersection)
            overlap = iou(box1, box2)
            
            # Also check if box2 is mostly inside box1
            x0_inter = max(x0_1, x0_2)
            y0_inter = max(y0_1, y0_2)
            x1_inter = min(x1_1, x1_2)
            y1_inter = min(y1_1, y1_2)
            
            if x1_inter > x0_inter and y1_inter > y0_inter:
                inter_area = (x1_inter - x0_inter) * (y1_inter - y0_inter)
                # Check if box2 is significantly inside box1 (at least 50% of box2 overlaps)
                if inter_area / area2 >= 0.5 or overlap >= iou_threshold:
                    used_panels2.add(j)
                    if verbose:
                        print(f"Model2 box {box2} overlaps with model1 box {box1} (IoU={overlap:.3f}), marking as used")
    
    # Add Blot panels from model2 that don't overlap with any model1 Blot panel
    for j, blot2 in enumerate(blot_panels2):
        if j not in used_panels2:
            processed_blot_panels.append(blot2)
            _, conf2, x0_2, y0_2, x1_2, y1_2 = blot2
            if verbose:
                print(f"Enriching with Blot from model2 (no overlap with model1): {(x0_2, y0_2, x1_2, y1_2)}")
    
    # Combine: non-Blot panels from model1 + processed Blot panels
    merged_panels = non_blot_panels1 + processed_blot_panels
    
    if verbose:
        print(f"Final merged panels: {len(merged_panels)} total ({len(non_blot_panels1)} non-Blot, {len(processed_blot_panels)} Blot)")
    
    return merged_panels

---
### 3.1 Secondary YOLO Model (960 px, blot-focused)

Higher input resolution with more aggressive NMS settings (`conf=0.3`, `iou=0.1`) to maximise
blot panel recall at the cost of more false positives — these are filtered by the merge logic above.
Graphs, Flow Cytometry, Body Imaging, and Microscopy panels are excluded from this model's output
since the primary 640 px model is sufficient for those categories.


In [4]:
yolo_blot_960 = PanelExtractor(
    weights_path="/kaggle/input/yolopanel/best_v10_yolo12_3.pt",
    device=device,
    img_size=960,
    conf_threshold=0.3,
    iou_threshold=0.1
)
yolo_blot_960.EXCLUDED_LABELS = {"Graphs", "Flow Cytometry", 'Body Imaging', "Microscopy"}

1. Detector loaded successfully from /kaggle/input/yolopanel/best_v10_yolo12_3.pt. Classes: {0: 'Blots', 1: 'Graphs', 2: 'Microscopy', 3: 'Body Imaging', 4: 'Flow Cytometry'}
Size: 960 | conf thresh: 0.3, IOU: 0.1, min crop side: 5 pixels


---
## 4. Fine-tuned Feature Extractor Weights (ALIKED + LightGlue for Blots)

The ALIKED keypoint extractor and LightGlue matcher used for blot-panel verification are
**fine-tuned end-to-end** on blot-specific training pairs (320×240 crops, 25,000 iterations).

**Why fine-tune?**
Pre-training on natural images produces keypoints biased toward texture edges and corners.
Fine-tuning on western blot crops redirects attention to band boundaries, lane separators, and
gel-lane texture patterns — the features that actually reveal copy-move operations in blot images.

The checkpoint stores both sub-modules under `extractor.*` and `matcher.*` key prefixes,
which are loaded into the respective model components separately.


In [5]:
import torch
weights = torch.load("/kaggle/input/neededpip/aliked_blot_320x240_25000it_0419.pth", map_location='cpu')['model']
print(matcher_blot.extractor.load_state_dict({key.replace('extractor.', ''):value for key, value in weights.items() if 'extractor.' in key}))
print(matcher_blot.matcher.load_state_dict({key.replace('matcher.', ''):value for key, value in weights.items() if 'matcher.' in key}))

<All keys matched successfully>
<All keys matched successfully>


---
## 5. Embedder Smoke Tests

Sanity-check all three embedding models on a sample western blot image pair before running full inference.
These tests verify that similarity scores fall in expected ranges and that model weights loaded correctly.


In [6]:
img1 = np.array(Image.open('/kaggle/input/overlapclf/wblot_sample.png').convert('RGB'))
img2 = np.array(Image.open('/kaggle/input/overlapclf/wblot_sample_sub2.png').convert('RGB'))

MICRO_DUP_SCORE_THRESH = 0.85
WBLOT_OVERLAP_PATH = "/kaggle/input/blotduplication/duplicate-supcon-epoch02-step10276-val_pairs_f10.85842-val_pairs_auc0.99835.ckpt"
WBLOT_OVERLAP_THRESHOLD = 0.85

wblot_overlap_embedder = UniversalEmbedderInference(WBLOT_OVERLAP_PATH, device='cuda', width=320, height=64, transform_type='albu_longest_max_size', head_type='v2', map_location='cpu')
print('wblot_overlap_embedder :', wblot_overlap_embedder.compare(img1,img2))

Using pooling type: avg
Model : nextvit_small.bd_ssld_6m_in1k. Using normalization mean: (0.485, 0.456, 0.406), std: (0.229, 0.224, 0.225)
using transform_type: albu_longest_max_size
wblot_overlap_embedder : 0.9631083607673645


In [7]:
img1 = np.array(Image.open('/kaggle/input/overlapclf/wblot_sample.png').convert('RGB'))
img2 = np.array(Image.open('/kaggle/input/overlapclf/wblot_sample_sub2.png').convert('RGB'))
print('micro_overlap_embedder :\n', micro_overlap_embedder.compare(img1,img2))

micro_overlap_embedder :
 0.9486770033836365


In [8]:
print(blot_duplicator_supcon.compare(img1,img2))

0.5912610292434692


---
## 6. Fallback Segmentor — DINOv2 Encoder + U-Net-style Decoder

When the geometric matching pipeline finds no candidate duplicate pairs, a **pixel-level
segmentation model** acts as a fallback to catch forgeries that do not follow a clean copy-move
pattern (e.g., brightness-adjusted duplicates, spliced panels from different sources).

### Architecture

```
Input (H×W×3 RGB image)
    │  resize to 518×518, normalise
    ▼
DINOv2-Base Encoder (frozen ViT-B/14)
    │  patch tokens → reshape to feature map
    ▼  [B, 768, 37, 37]
┌───────────────────────────────────────────┐
│  Decoder (U-Net-style progressive upsamp) │
│                                           │
│  Block 1: Conv2d(768→384) + ReLU + Drop   │  → [B, 384, 74, 74]
│  Block 2: Conv2d(384→192) + ReLU + Drop   │  → [B, 192, 148, 148]
│  Block 3: Conv2d(192→96)  + ReLU          │  → [B, 96, 296, 296]
│  Head:    Conv2d(96→1)  + bilinear resize  │  → [B, 1, H, W]
└───────────────────────────────────────────┘
    │
Sigmoid → probability map → post-processing → binary mask
```

### Design Rationale

- **Frozen DINOv2 encoder** — self-supervised ViT features provide strong patch-level semantics
  that transfer well to biomedical images without requiring a large labelled forgery corpus.
- **Lightweight decoder (~1.5 M trainable parameters)** — avoids overfitting on the ~3,000
  training images; only the decoder is trained.
- **Dropout2d in early decoder blocks** — acts as structured regularisation on spatial feature maps,
  improving generalisation across different panel types and imaging modalities.

### Post-processing Pipeline

```
prob_map (518×518 float32)
    │
    ├─ Sobel gradient magnitude (x + y) → normalised grad_norm
    │
    ├─ enhanced = 0.55 * prob + 0.45 * grad_norm  (sharpens forgery boundaries)
    │
    ├─ Gaussian blur (3×3) to suppress noise
    │
    ├─ Adaptive threshold: μ(enhanced) + 0.3 * σ(enhanced)
    │
    ├─ Morphological CLOSE (5×5 kernel) → fills small holes in forgery region
    │
    ├─ Morphological OPEN  (3×3 kernel) → removes isolated noise pixels
    │
    ├─ Resize to original image dimensions (nearest-neighbour)
    │
    └─ Authenticity gate:
         if area < 200 px OR mean_prob_inside < 0.22 → predict "authentic"
```

### Test-Time Augmentation (TTA)

Optional horizontal + vertical flip averaging (`USE_TTA=False` by default) produces more
stable probability maps at the cost of 3× inference time.


In [9]:
import os, cv2, json, math, random, torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from transformers import AutoImageProcessor, AutoModel


def rle_encode_custom(mask: np.ndarray, fg_val: int = 1) -> str:
    pixels = mask.T.flatten()
    dots = np.where(pixels == fg_val)[0]
    if len(dots) == 0:
        return "authentic"
    run_lengths = []
    prev = -2
    for b in dots:
        if b > prev + 1:
            run_lengths.extend((b + 1, 0))
        run_lengths[-1] += 1
        prev = b
    return json.dumps([int(x) for x in run_lengths])

BASE_DIR  = "/kaggle/input/recodai-luc-scientific-image-forgery-detection"
AUTH_DIR  = f"{BASE_DIR}/train_images/authentic"
FORG_DIR  = f"{BASE_DIR}/train_images/forged"
MASK_DIR  = f"{BASE_DIR}/train_masks"
TEST_DIR  = f"{BASE_DIR}/test_images"
DINO_PATH = "/kaggle/input/dinov2/pytorch/base/1"

IMG_SIZE = 518
MODEL_LOC = '/kaggle/input/cnndinov2-pbd/CNNDINOv2-U52/CNNDINOv2-U52/model_seg_final.pt'  # 0.321

# INFERENCE UTILS
AREA_THR = 200
MEAN_THR = 0.22
USE_TTA = False
GRID_SEARCH = False


from transformers import AutoImageProcessor, AutoModel
processor = AutoImageProcessor.from_pretrained(DINO_PATH, local_files_only=True, use_fast=False)
encoder = AutoModel.from_pretrained(DINO_PATH, local_files_only=True).eval().to(device)

class DinoTinyDecoder(nn.Module):
    def __init__(self, in_ch=768, out_ch=1):
        super().__init__()
        # Block 1: 768 -> 384
        self.block1 = nn.Sequential(
            nn.Conv2d(in_ch, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )
        # Block 2: 384 -> 192
        self.block2 = nn.Sequential(
            nn.Conv2d(384, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Dropout2d(0.1)
        )
        # Block 3: 192 -> 96
        self.block3 = nn.Sequential(
            nn.Conv2d(192, 96, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
        # Final Output: 96 -> 1
        self.conv_out = nn.Conv2d(96, out_ch, kernel_size=1)
    
    def forward(self, f, target_size):
        # f: [B, 768, 37, 37]
        
        # Step 1: Up to ~74x74
        x = F.interpolate(self.block1(f), size=(74, 74), mode='bilinear', align_corners=False)
        
        # Step 2: Up to ~148x148
        x = F.interpolate(self.block2(x), size=(148, 148), mode='bilinear', align_corners=False)
        
        # Step 3: Up to ~296x296
        x = F.interpolate(self.block3(x), size=(296, 296), mode='bilinear', align_corners=False)
        
        # Step 4: Final jump to 518x518
        x = self.conv_out(x)
        x = F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)
        
        return x
    
class DinoSegmenter(nn.Module):
    def __init__(self, encoder, processor):
        super().__init__()
        self.encoder, self.processor = encoder, processor
        for p in self.encoder.parameters(): p.requires_grad = False
        self.seg_head = DinoTinyDecoder(768,1)
    def forward_features(self,x):
        imgs = (x*255).clamp(0,255).byte().permute(0,2,3,1).cpu().numpy()
        inputs = self.processor(images=list(imgs), return_tensors="pt").to(x.device)
        # with torch.no_grad(): 
        #     feats = self.encoder(**inputs).last_hidden_state
        feats = self.encoder(**inputs).last_hidden_state
        B,N,C = feats.shape
        fmap = feats[:,1:,:].permute(0,2,1)
        s = int(math.sqrt(N-1))
        fmap = fmap.reshape(B,C,s,s)
        return fmap
    def forward_seg(self,x):
        fmap = self.forward_features(x)
        return self.seg_head(fmap,(IMG_SIZE,IMG_SIZE))


model_seg = DinoSegmenter(encoder, processor).to(device)

# Load pretrained weights if MODEL_LOC is specified
if MODEL_LOC is not None and os.path.exists(MODEL_LOC):
    model_seg.load_state_dict(torch.load(MODEL_LOC, map_location=device))
    print(f"✅ Loaded pretrained model from: {MODEL_LOC}")
    model_seg.eval()  # Set model to evaluation mode

@torch.no_grad()
def segment_prob_map(pil):
    x = torch.from_numpy(np.array(pil.resize((IMG_SIZE, IMG_SIZE)), np.float32)/255.).permute(2,0,1)[None].to(device)
    prob = torch.sigmoid(model_seg.forward_seg(x))[0,0].cpu().numpy()
    return prob

@torch.no_grad()
def segment_prob_map_with_tta(pil):
    # 1. Preprocessing: Resize, Normalize, and move to Device
    x = torch.from_numpy(np.array(pil.resize((IMG_SIZE, IMG_SIZE)), np.float32)/255.).permute(2,0,1)[None].to(device)
    
    predictions = []

    # 2. Original Prediction
    pred_orig = torch.sigmoid(model_seg.forward_seg(x))
    predictions.append(pred_orig)

    # 3. Horizontal Flip TTA (dim 3)
    # Flip input -> Predict -> Flip output back
    pred_h = torch.sigmoid(model_seg.forward_seg(torch.flip(x, dims=[3])))
    predictions.append(torch.flip(pred_h, dims=[3]))

    # 4. Vertical Flip TTA (dim 2)
    # Flip input -> Predict -> Flip output back
    pred_v = torch.sigmoid(model_seg.forward_seg(torch.flip(x, dims=[2])))
    predictions.append(torch.flip(pred_v, dims=[2]))

    # 5. Average the predictions and format as numpy
    # We stack the 3 predictions and take the mean across the stack dimension (0)
    prob = torch.stack(predictions).mean(0)[0, 0].cpu().numpy()

    return prob
    
def enhanced_adaptive_mask(prob, alpha_grad=0.45):
    gx = cv2.Sobel(prob, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(prob, cv2.CV_32F, 0, 1, ksize=3)
    grad_mag = np.sqrt(gx**2 + gy**2)
    grad_norm = grad_mag / (grad_mag.max() + 1e-6)
    enhanced = (1 - alpha_grad) * prob + alpha_grad * grad_norm
    enhanced = cv2.GaussianBlur(enhanced, (3,3), 0)
    thr = np.mean(enhanced) + 0.3 * np.std(enhanced)
    mask = (enhanced > thr).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, np.ones((5,5), np.uint8))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    return mask, thr

def finalize_mask(prob, orig_size):
    mask, thr = enhanced_adaptive_mask(prob)
    mask = cv2.resize(mask, orig_size, interpolation=cv2.INTER_NEAREST)
    return mask, thr

def pipeline_final(pil):
    if USE_TTA:
        prob = segment_prob_map_with_tta(pil)
    else:
        prob = segment_prob_map(pil)
    mask, thr = finalize_mask(prob, pil.size)
    area = int(mask.sum())
    mean_inside = float(prob[cv2.resize(mask,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_NEAREST)==1].mean()) if area>0 else 0.0
    if area < AREA_THR or mean_inside < MEAN_THR:
        return "authentic", None, {"area": area, "mean_inside": mean_inside, "thr": thr}
    return "forged", mask, {"area": area, "mean_inside": mean_inside, "thr": thr}


def get_segmentator_mask(path, skip_blot=False):
    image = Image.open(path).convert("RGB")
    # label, mask = infer_image(image)
    label, mask, dbg = pipeline_final(image)
    if mask is None:
        return np.zeros(image.size[::-1], np.uint8)
    try:
        mask = np.array(mask, dtype=np.uint8)
        img = PanelExtractor._load_image(path)
        panels = extractor.extract_panels(img)
        region_mask = np.zeros_like(mask, dtype=np.uint8)
        
        for label, _, x1, y1, x2, y2 in panels:
            if skip_blot and label == 'Blots':
                print('skip panel because blot')
                continue
            # Round coordinates to nearest integer and clamp to mask size
            x1 = max(0, min(mask.shape[1]-1, int(round(x1))))
            y1 = max(0, min(mask.shape[0]-1, int(round(y1))))
            x2 = max(0, min(mask.shape[1], int(round(x2))))
            y2 = max(0, min(mask.shape[0], int(round(y2))))
            
            # Set region to 1
            region_mask[y1:y2, x1:x2] = 1
        
        # Apply region mask to original mask
        mask_SEG = mask * region_mask
    except Exception as ex:
        print(ex)
        return np.zeros(image.size[::-1], np.uint8)
    return mask_SEG

E0000 00:00:1768375844.839163      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768375844.929153      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


✅ Loaded pretrained model from: /kaggle/input/cnndinov2-pbd/CNNDINOv2-U52/CNNDINOv2-U52/model_seg_final.pt


---
## 7. Strip Detection Module (Blot-Specific)

Western blots are sometimes manipulated by splicing **horizontal strips** from different
exposure runs rather than duplicating entire panels. Standard copy-move detection misses these
because each strip may appear only once in the full image.

### Sub-pipeline

1. **YOLOv8-Seg** (`seg_v8_best.pt`) — instance segmentation to isolate individual bands/strips
   within each detected blot panel
2. **Strip Similarity Embedder** — 128×128 crops compared pairwise via a SupCon-trained
   embedder (`SEG_SIM_THRESH=0.75`)

**Activation condition:** strip detection runs only when:
- At least one blot panel is detected, AND
- The main embedding pipeline found **no** duplicate blot pairs (avoids double-counting)

When strip matches are found, `create_strip_match_masks` generates pixel-level masks
for the matched strip regions.


In [10]:
from ultralytics import YOLO
segmentator = YOLO("/kaggle/input/yolopanel/seg_v8_best.pt")

SEG_SIM_THRESH = 0.75
wblot_segment_embedder = UniversalEmbedderInference("/kaggle/input/blotduplication/strip_dup_kaggle/v6_box-duplicate-detect-epoch00-step8000-val_loss0.08147.ckpt", device='cuda', width=128, height=128, transform_type='albu_longest_max_size', head_type='v2', map_location='cpu')

Using pooling type: avg
Model : nextvit_small.bd_ssld_6m_in1k. Using normalization mean: (0.485, 0.456, 0.406), std: (0.229, 0.224, 0.225)
using transform_type: albu_longest_max_size


In [11]:
VERBOSE = False

---
## 8. Main Inference Loop

End-to-end inference over the full test set. Decision logic per image:

```
For each test image:

  1. Load image → run dual-YOLO panel detection
  2. Compute panel intersections (overlapping panels → skip pair to avoid FP)

  ── BLOT PANELS ──
  3a. Batch-embed all blot crops with:
        - wblot_overlap_embedder  (threshold=0.85) — overlap/partial duplicate
        - blot_duplicator_supcon  (threshold=0.84) — full duplicate
  3b. Merge candidate pairs (keep highest score per unique panel pair)

  ── MICROSCOPY PANELS ──
  4.  Batch-embed all microscopy crops (ensemble, threshold=0.58)

  5.  Remove pairs whose panels geometrically intersect

  ── GEOMETRIC VERIFICATION ──
  6.  Run LightGlue on each candidate pair:
        Blots    → always keep ALIKED match results (blot forgeries are subtle)
        Microscopy:
          good inliers + good score  → KEEP
          good inliers + low score   → re-verify with micro_overlap_embedder
          low inliers OR low score   → SKIP

  7.  Merge accepted masks via max-clique grouping
      (handles chains A~B~C → single merged region)

  ── FALLBACK PIPELINE ──
  8.  If no matcher results → run strip detector on blot panels
  9.  If still no results  → DINOv2 fallback segmentor (blot panels excluded)

  10. Encode final mask as RLE  OR  return "authentic"
```

### Competition Metric: oF1 Score

The evaluation metric is **object-level F1**: a predicted mask is a true positive only if its
IoU with the corresponding ground-truth mask exceeds 0.1. This rewards finding the right *region*
over pixel-perfect boundaries, but heavily penalises spurious false-positive masks — which motivates
the conservative similarity thresholds and the two-stage (embedding → geometric) filtering strategy.

> **Final score: 0.550 private oF1 / 0.450 public oF1.**


In [12]:
result = []
for path in tqdm(glob('/kaggle/input/recodai-luc-scientific-image-forgery-detection/test_images/*')):
# for path in tqdm(sorted(glob("/kaggle/input/recodai-luc-scientific-image-forgery-detection/supplemental_images/*"))):    
    case_id = int(path.split('/')[-1].split('.')[0])
    print(case_id)
    try:
        img = PanelExtractor._load_image(path)
        height, width = img.shape[:2]
        # panels = extractor.extract_panels(img)  # model 1 - YOLO
        panels = merge_dual_model_detections(
            extractor1=extractor,
            extractor2=yolo_blot_960,
            img=img,
            iou_threshold=0.25,  # IoU threshold for considering overlaps
            verbose=False         # Set to True for debug output
        )
        intersections = get_intersections(panels, margin=10)   ############### <<<<<<<<<<<<<<<<<<< TODO 1: ДЕТАЛЬНО ПРОДЕБАЖИТЬ
    
        crops_list = PanelExtractor.crop_panels(img, panels)   ############### <<<<<<<<<<<<<<<<<<< TODO 2: энивей пока нет модели, но ЧТО ЕСЛИ НЕТ ПАНЕЛЕЙ или ПАНЕЛЬ ОДНА И *ПОЧТИ* НА ВСЮ ФОТКУ --> ИСПОЛЬЗУЕМ ВСЮ ФОТКУ?
    
        blot_panels_ids = [i for i in range(len(panels)) if panels[i][0] == 'Blots']
        microscopy_panels_ids = [i for i in range(len(panels)) if panels[i][0] == 'Microscopy']
    
        if VERBOSE:
            print(f'case : {case_id} | num panels : {len(panels)} | num blots panels : {len(blot_panels_ids)} num micro panels : {len(microscopy_panels_ids)}')
    
        ######### COMPUTE SIMILARITIES FOR WBLOT:
        if len(blot_panels_ids):
            crops_list_blot = PanelExtractor.crop_panels(img, [panels[i] for i in blot_panels_ids])
            embeddings_blot_overlaper = wblot_overlap_embedder.get_embedding_batch(crops_list_blot).cpu()
            pairs_overlaper_filtered = [
                (
                    blot_panels_ids[i],  # source index in panels
                    blot_panels_ids[j],
                    score
                )
                for i, j, score in UniversalEmbedderInference.find_similar_pairs(embeddings_blot_overlaper, threshold=WBLOT_OVERLAP_THRESHOLD)  
            ]
            embeddings_duplicate_detetector = blot_duplicator_supcon.get_embedding_batch(crops_list_blot).cpu()
            pairs_duplicate_detetector = [
                (
                    blot_panels_ids[i],  # source index in panels
                    blot_panels_ids[j],
                    score
                )
                for i, j, score in UniversalEmbedderInference.find_similar_pairs(embeddings_duplicate_detetector, threshold=WBLOT_DUP_SCORE_THRESH1)  
            ]
            print('pairs_overlaper_filtered :', len(pairs_overlaper_filtered), 'pairs_duplicate_detetector :', len(pairs_duplicate_detetector))
            pairs_dict = {}
            for i, j, score in pairs_overlaper_filtered + pairs_duplicate_detetector:
                key = tuple(sorted((i, j)))  # Normalize pair order
                if key not in pairs_dict or score > pairs_dict[key]:
                    pairs_dict[key] = score
            
            similar_pairs_blot = [
                ('Blots', score, i, j)
                for (i, j), score in pairs_dict.items()
            ]
        else:
            similar_pairs_blot = []
    
        ######### COMPUTE SIMILARITIES FOR MICROSCOPY:
        if len(microscopy_panels_ids):
            crops_list_microscopy = PanelExtractor.crop_panels(img, [panels[i] for i in microscopy_panels_ids])
            embeddings_microscopy = micro_overlap_embedder.get_embedding_batch(crops_list_microscopy).cpu()
            similar_pairs_microscopy = [
                (
                    'Microscopy',
                    score,
                    microscopy_panels_ids[i],  # source index in panels
                    microscopy_panels_ids[j]
                )
                for i, j, score in UniversalEmbedderInference.find_similar_pairs(embeddings_microscopy, threshold=MICROSCOPY_EMB_THRESH)
            ]
        else:
            similar_pairs_microscopy = []
        
        similar_pairs = similar_pairs_blot + similar_pairs_microscopy    
        similar_pairs[:] = [
            (label, score, i, j)
            for (label, score, i, j) in similar_pairs
            if (i, j) not in intersections and (j, i) not in intersections
        ]
    
        clf_predicts = pd.DataFrame(similar_pairs, columns=['label', 'score', 'idx1', 'idx2'])

        match_results = create_duplicate_masks_v5(img, panels, crops_list, clf_predicts, matcher_micro, matcher_blot, to_bbox_micro=False, to_bbox_blot=True, fallback_for_wblot=True, 
                                                 test_transforms_blot=False, test_transforms_micro=True)
        print('valid results from matcher :', len(match_results))
            
        pred_masks, duplicate_info = [], []
        for info in match_results:
            id0 = info['panel_id0']
            id1 = info['panel_id1']
            label = info['panel_label']
            match_result = info['match_result']
            inliers = match_result["inliers"]
            matcher_model_score = match_result[match_filter_str]
            if label == 'Blots':
                if inliers < inlier_threshold or matcher_model_score < match_score_threshold:
                    print(f"-----4. Keep BLOT pair ({id0}, {id1}): {inliers} inliers, {matcher_model_score:.3f} with label Blots")
                instance_mask = (info['mask0'] | info['mask1']).astype(np.uint8)
                pred_masks.append(instance_mask)
                duplicate_info.append(info)
                continue
        
            # Microscopy filtering logic
            if (inliers >= inlier_threshold) and (matcher_model_score < match_score_threshold):
                # Only one condition is bad (good inliers, bad score) - check micro deduplicator
                bbox_crop0 = info['bbox_crop0']
                bbox_crop1 = info['bbox_crop1']
                img0 = crops_list[id0]
                img1 = crops_list[id1]
                img0_crop = Image.fromarray(img0).crop(bbox_crop0)
                img1_crop = Image.fromarray(img1).crop(bbox_crop1)
                micro_score = micro_overlap_embedder.compare(img0_crop, img1_crop)
                
                if micro_score < MICRO_DUP_SCORE_THRESH:
                    print(f"-----1. Skipping pair ({id0}, {id1}) due to low match score + low sim score: {matcher_model_score:.3f} AND MicroDuplicate sim score: {micro_score:.3f}. BTW: inliers={inliers}")
                    continue
                else:
                    print(f"-----2. Keep pair ({id0}, {id1}): inliers={inliers}; match score {matcher_model_score:.3f}(LOW) +++ MicroDuplicate sim score: {micro_score:.3f} (HIGH)")
            elif (inliers < inlier_threshold) or (matcher_model_score < match_score_threshold):
                print(f"-----3. Skipping pair ({id0}, {id1}) due to low {inliers} inliers or matcher_model_score: {matcher_model_score:.3f}.")
                continue
    
            instance_mask = (info['mask0'] | info['mask1']).astype(np.uint8)
            pred_masks.append(instance_mask)
            duplicate_info.append(info)        
        mask_MATCHER, merged_info = merge_masks_by_max_cliques_v5(pred_masks, duplicate_info, verbose=VERBOSE)
    except Exception as ex:
        print(ex)
        mask_MATCHER = []

    
    mask_STRIPS = []
    if len(blot_panels_ids) > 0 and not len(similar_pairs_blot):
        strip_match_result = find_strips_in_blot_panels(
            panels=panels,
            blot_panels_ids=blot_panels_ids,
            crops_list=crops_list,
            segmentator=segmentator,
            blot_duplicate_detector=wblot_segment_embedder,
            similarity_threshold=SEG_SIM_THRESH,
            overlap_threshold=5
        )
        if len(strip_match_result):
            mask_STRIPS = create_strip_match_masks(img.shape, strip_match_result['best_matches'], strips=strip_match_result['strips'])
            print('~~~~~~~~~~~~~~~~~~~', len(mask_STRIPS))
            
    mask_MATCHER = mask_MATCHER + mask_STRIPS

    if not len(mask_MATCHER):
        # NOTE: segmentator is strict fallback; do NOT combine with matcher outputs
        mask_SEG = get_segmentator_mask(path, skip_blot=True)
        if mask_SEG.sum() == 0:
            annotation = "authentic"
        else:
            annotation = rle_encode_custom((mask_SEG > 0).astype(np.uint8))  # use public notebook
    else:
        annotation = rle_encode(mask_MATCHER)   # NOT doing combine like mask_MATCHER + [(mask_SEG > 0).astype(np.uint8)]
    
    result.append({
        "case_id": case_id,
        "annotation": annotation
    })

  0%|          | 0/1 [00:00<?, ?it/s]

45
pairs_overlaper_filtered : 1 pairs_duplicate_detetector : 1
valid results from matcher : 1


100%|██████████| 1/1 [00:03<00:00,  3.59s/it]


---
## 9. Output & Submission

Collect all per-image predictions into a DataFrame and write `submission.csv`.

### Output Format

| Value | Meaning |
|-------|---------|
| `"authentic"` | No forgery detected in this image |
| `[start, length, start, length, ...]` | JSON-encoded RLE of the binary forgery mask (column-major order) |

The RLE encoding is column-major (transposed mask flattened) to match the competition specification.


In [13]:
result = pd.DataFrame(result, columns=['case_id', 'annotation'])

In [14]:
result.to_csv('submission.csv', index=False)

In [15]:
result

,case_id,annotation
0,45,"[441742, 88, 443186, 88, 444630, 88, 446074, 8..."


In [16]:
(result.annotation == 'authentic').sum()

0

In [17]:
clf_predicts

,label,score,idx1,idx2
0,Blots,0.974576,3,4
